# CrewAI Agents with Structured Output

In [1]:
!pip install -q crewai crewai_tools

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 2.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.6/90.6 kB 5.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 kB 2.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.1/69.1 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 36.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 189.8/189.8 kB 10.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 820.8/820.8 kB 35.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.0/85.0 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.9/19.9 MB 80.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 252.5/252.5 kB 14.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.0/48.0 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4

In [2]:
import crewai
import crewai_tools

print(crewai.__version__)
print(crewai_tools.__version__)

1.15.10
1.15.10


# Set API Keys

In [3]:
from google.colab import userdata
import os
os.environ["SERPER_API_KEY"] = userdata.get('SERPER_API_KEY')
os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY_NEW')
os.environ["GROQ_API_KEY"] = userdata.get('GROQ_API_KEY')


# Import Dependencies

In [4]:

from crewai import Agent, Crew, Task, Process, LLM, Process
from crewai_tools import SerperDevTool, DirectoryReadTool
from pydantic import BaseModel
from typing import List

# from dotenv import load_dotenv
# load_dotenv()

#------------------------------------------------------------------------
import warnings
warnings.filterwarnings("ignore")
#------------------------------------------------------------------------


# Define LLMs

In [5]:
# Create an LLM with a temperature of 0 to ensure deterministic outputs
# -----------
# Create LLM
# -----------

# Create an LLM with a temperature of 0 to ensure deterministic outputs
# OPENAI LLMs
llm = LLM(
         # model="gpt-5.4-mini",
          model="gpt-5.4-nano",
          base_url="https://api.openai.com/v1",
          api_key = os.environ["OPENAI_API_KEY"],
          temperature=0.2)

# # GROQ hosted LLMs
# llm = LLM(
#      model="llama-3.3-70b-versatile",
#      base_url="https://api.groq.com/openai/v1",
#      api_key=os.environ["GROQ_API_KEY"],
#      temperature=0.4)



# Tools

In [6]:
#------------------------------------------------------------------------
# Create tools
search_tool = SerperDevTool()  # Search capability
docs_tool = DirectoryReadTool(directory='./blog-posts')  # Reads from local files

# Define Output Classes

In [7]:
#------------------------------------------------------------------------
# Define the Output Class to ensure Structured output from the crew
# This will be used to validate the output of the tasks

class ResearchFindings(BaseModel):
    main_points: List[str]
    key_technologies: List[str]
    societal_impact: str

class Report(BaseModel):
    title: str
    introduction: str
    body: str
    conclusion: str
#------------------------------------------------------------------------


# Define Agents

In [8]:
# Create Agents
researcher = Agent(
    role='Research Analyst',
    goal='Use available tool to collect information and provide up-to-date technical and social analysis on a given topic',
    backstory='An expert analyst with a keen eye for technical nitty-gritty with a perspective on human vakues.',
    tools=[search_tool],
    llm=llm,
    verbose=False
)

writer = Agent(
    role='Content Writer',
    goal='Craft engaging report about the provided topic',
    backstory='A skilled writer with a passion for technology and its impact on humanity.',
    tools=[docs_tool],
    llm=llm,
    verbose=False
)


# Define Tasks

In [9]:
#------------------------------------------------------------------------
# Define tasks
research_task = Task(
    description='Research the latest trends in the topic {topic}',
    expected_output=('A summary of recent developments including a unique perspective on their significance.'
        'Your output should contain the following:'
        'main_points: the main textual summary of the report'
        'key_technologies: key technologies enabling the change'
        'societal_impact: how it impacts the life of people and the society as a whole'),
    agent=researcher,
    output_pydantic = ResearchFindings
)

writing_task = Task(
    description=("""Write an engaging report about a topic based on the research analyst's summary.
                    You will receive research output in JSON format from the researcher.
                    You need to extract the following piece of information from the object returned by the `research_task`
                    'main_points: the main textual summary of the report'
                    'key_technologies: key technologies enabling the change'
                    'societal_impact: how it impacts the life of people and the society as a whole'
                     'Use this information to write a comprehensive and engaging report.'
                     """),
    expected_output=(
        "A structured report with title, introduction, body, and conclusion, "
        "written in a clear and engaging style."),
    agent=writer,
    output_pydantic=Report,       # Structured output format
    output_file='blog-posts/report.md',  # The final blog post will be saved here
    context = [research_task], # Pass research context to writer
    verbose=True
)

# Define Crew (Orchestration Layer)

In [10]:
#-----------------------------------------------------------------------------
# Assemble a crew with planning enabled
crew = Crew(
    agents=[researcher, writer],
    tasks=[research_task, writing_task],
    verbose=False,
    process=Process.sequential,
    planning=True,  # Enable planning feature
)

# Run the Crew

In [11]:
# Run the Crew
results = await crew.kickoff_async(
    inputs={"topic": "Social media and its impact on humans"}
    )


In [12]:
last_result = results.pydantic
last_result
# from pprint import pprint
# pprint(last_result)

Report(title='When Social Media Becomes an Adaptive Behavioral System: Balancing Connection, Commerce, and Human Well-Being', introduction='Social media is evolving rapidly because the platforms behind it are changing how content is produced, selected, and delivered—and how users interact with it moment by moment. Recent developments span platform behavior and product changes, shifts in user habits and attention/screen-time patterns, and new content formats (especially short-form video). At the same time, AI-generated and AI-curated content, recommendation algorithms, social commerce, private/community-based sharing, and evolving moderation and safety tools are reshaping what people see, how long they stay engaged, and what kinds of social experiences they encounter. These changes matter because they directly influence human outcomes such as mental health, loneliness, anxiety, self-esteem, misinformation exposure, harassment, and political polarization—while also affecting broader soci

In [13]:
#------------------------------------------------------------
# Verify successful context passing between the agents
#------------------------------------------------------------

# Extract the outputs of individual tasks
research_output = results.tasks_output[0].pydantic
writing_output = results.tasks_output[1].pydantic

# --- Verification Logic ---
# Check if the research output is a valid Pydantic model
if not isinstance(research_output, ResearchFindings):
    print("❌ Research task did not produce a valid ResearchFindings object.")
else:
    print("✅ Research task produced a valid ResearchFindings object.")

    # Get a specific piece of information from the research findings
    # For example, the first main point or a key technology
    key_research_point = research_output.main_points[0] if research_output.main_points else ""
    key_technology = research_output.key_technologies[0] if research_output.key_technologies else ""

    print(f"\nKey research point to check: '{key_research_point}'")
    print(f"\nKey technology to check: '{key_technology}'")

    # Check if the writing output is a valid Pydantic model
    if not isinstance(writing_output, Report):
        print("❌ Writing task did not produce a valid Report object.")
    else:
        print("✅ Writing task produced a valid Report object.")

        # Now, verify if the writer's report contains the information from the researcher
        # This is the core of the verification
        if key_research_point in writing_output.body or key_research_point in writing_output.introduction:
            print("🎉 SUCCESS: The writer's report successfully incorporated the research context!")
        else:
            print("⚠️ The writer's report seems to be missing the key research context.\n\n")



╭────────────────────────── Tracing Preference Saved ──────────────────────────╮
│                                                                              │
│  Info: Tracing has been disabled.                                            │
│                                                                              │
│  Your preference has been saved. Future Crew/Flow executions will not        │
│  collect traces.                                                             │
│                                                                              │
│  To enable tracing later, do any one of these:                               │
│  • Set tracing=True in your Crew/Flow code                                   │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file               │
│  • Run: crewai traces enable                                                 │
│                                                                              │
╰───────────────────────────

# Research Task Output

In [45]:
# Extract the outputs of individual tasks
research_output = results.tasks_output[0] # index 0 denotes the first task i.e research_task

# Option 1: Accessing Properties Directly from the Pydantic Model
print("\n\n Accessing Properties - Option 1")
main_points = research_output.pydantic.main_points
key_technologies = research_output.pydantic.key_technologies
societal_impact = research_output.pydantic.societal_impact

print("Main Points:", main_points)
print("Key Technologies:", key_technologies)
print("Societal Impact:", societal_impact)

#--------------------------------------------------------------------

# Option 2: Accessing Properties Using the to_dict() Method
print("\n\n Accessing Properties - Option 2")
output_dict = research_output.to_dict()
main_points = output_dict["main_points"]
key_technologies = output_dict["key_technologies"]
societal_impact = output_dict["societal_impact"]

print("Main Points:", main_points)
print("Key Technologies:", key_technologies)
print("Societal Impact:", societal_impact)

#--------------------------------------------------------------------

# Option 3: Printing the Entire Object
print("\n\n Accessing Properties - Option 3")
print("Research Output:", research_output)



 Accessing Properties - Option 1
Main Points: ['Scope (precise): This research focuses on the latest trends in social media and their impact on humans, emphasizing recent developments in (1) platform behavior and product changes, (2) user habits and attention/screen-time patterns, (3) content formats (especially short-form video), (4) AI-generated and AI-curated content, (5) recommendation algorithms and ranking systems, (6) social commerce and creator/influencer economy shifts, (7) private/community-based sharing, (8) moderation and safety tools, and (9) changes in youth usage patterns—then connects these to human outcomes (mental health, loneliness, anxiety, self-esteem, misinformation exposure, harassment, political polarization) and broader societal effects (community building, entrepreneurship, public discourse, and institutional trust).', 'Short-form video dominance is a confirmed long-term trend: platforms continue to prioritize algorithmic feeds and mobile-first video formats

# Writing Task Output

In [49]:
# Extract the outputs of individual tasks
writing_output = results.tasks_output[1] # index 1 denotes the second/last task i.e writing_task

# Option 1: Accessing Properties Directly from the Pydantic Model
print("\n\n Accessing Properties - Option 1")
title = writing_output.pydantic.title
introduction = writing_output.pydantic.introduction
body = writing_output.pydantic.body
conclusion = writing_output.pydantic.conclusion

print("Title:", title)
print("Introduction:", introduction)
print("Body:", body)
print("Conclusion:", conclusion)

#--------------------------------------------------------------------

# Option 2: Accessing Properties Using the to_dict() Method
print("\n\n Accessing Properties - Option 2")
output_dict = writing_output.to_dict()
title = output_dict["title"]
introduction = output_dict["introduction"]
body = output_dict["body"]
conclusion = output_dict["conclusion"]


print("Title:", title)
print("Introduction:", introduction)
print("Body:", body)
print("Conclusion:", conclusion)

#--------------------------------------------------------------------

# Option 3: Printing the Entire Object
print("\n\n Accessing Properties - Option 3")
print("Research Output:", writing_output)



 Accessing Properties - Option 1
Title: When Social Media Becomes an Adaptive Behavioral System: Balancing Connection, Commerce, and Human Well-Being
Introduction: Social media is evolving rapidly because the platforms behind it are changing how content is produced, selected, and delivered—and how users interact with it moment by moment. Recent developments span platform behavior and product changes, shifts in user habits and attention/screen-time patterns, and new content formats (especially short-form video). At the same time, AI-generated and AI-curated content, recommendation algorithms, social commerce, private/community-based sharing, and evolving moderation and safety tools are reshaping what people see, how long they stay engaged, and what kinds of social experiences they encounter. These changes matter because they directly influence human outcomes such as mental health, loneliness, anxiety, self-esteem, misinformation exposure, harassment, and political polarization—while a